# Transformers e Aplicações de Perguntas e Respostas (questions-answers)

Nesse caderno, nós vamos explorar o uso de modelos baseados em Transformers para implementação de aplicações de Pertguntas e Respostas. Nós vamos utilizar o modelo pré-treinado [Bertimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased), refinado para cenários de extração de respostas de contextos.

Nesse exemplo, nós vamos utilizar pipelines pré-configurados do modelo que tem como objetivo facilitar o uso do modelo, abstraindo o uso de Tokenizers.

Inicialmente, nós vamos carregar o pipeline do modelo pré-treinado. Nesse pipeline, vamos identificar que estamos carregando um modelo de perguntas-respostas (question-answering), o nome do [modelo](https://huggingface.co/pierreguillou/bert-base-cased-squad-v1.1-portuguese) e o framework de back-end que vamos utilizar (TF, para o TensorFlow).

Após o carregamento do modelo, nós podemos utilizá-lo para aplicações de pergunta e resposta. Esse modelo deve receber os seguintes parâmetros como entrada:
1. Contexto;
2. Pergunta.

Nesse cenário, o modelo tem como objetivo identificar a resposta da pergunta dentro do contexto passados como parâmetros. A seguir, temos um exemplo de pergunta, contexto e resposta gerada pelo modelo.

In [ ]:
from transformers import pipeline


# https://huggingface.co/pierreguillou/bert-base-cased-squad-v1.1-portuguese
# source: https://pt.wikipedia.org/wiki/Pandemia_de_COVID-19
model_name = 'pierreguillou/bert-base-cased-squad-v1.1-portuguese'
nlp = pipeline("question-answering", model=model_name, framework='tf')

context = r"""
A pandemia de COVID-19, também conhecida como pandemia de coronavírus, é uma pandemia em curso de COVID-19,
uma doença respiratória aguda causada pelo coronavírus da síndrome respiratória aguda grave 2 (SARS-CoV-2).
A doença foi identificada pela primeira vez em Wuhan, na província de Hubei, República Popular da China,
em 1 de dezembro de 2019, mas o primeiro caso foi reportado em 31 de dezembro do mesmo ano.
Acredita-se que o vírus tenha uma origem zoonótica, porque os primeiros casos confirmados
tinham principalmente ligações ao Mercado Atacadista de Frutos do Mar de Huanan, que também vendia animais vivos.
Em 11 de março de 2020, a Organização Mundial da Saúde declarou o surto uma pandemia. Até 8 de fevereiro de 2021,
pelo menos 105 743 102 casos da doença foram confirmados em pelo menos 191 países e territórios,
com cerca de 2 308 943 mortes e 58 851 440 pessoas curadas.
"""

question = "Quando começou a pandemia de Covid-19 no mundo?"

result = nlp(question=question, context=context)

print(f"Answer: '{result['answer']}', score: {round(result['score'], 4)}, start: {result['start']}, end: {result['end']}")



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/862 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/434M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForQuestionAnswering.

All the layers of TFBertForQuestionAnswering were initialized from the model checkpoint at pierreguillou/bert-base-cased-squad-v1.1-portuguese.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForQuestionAnswering for predictions without further training.


tokenizer_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Answer: '1 de dezembro de 2019', score: 0.713, start: 325, end: 346


Observem que nesse caso o modelo identifica a resposta corretamente. Na resposta do modelo, também é identificada uma pontuação (índice de confiança) para a resposta do modelo e a posição da resposta identificada no contexto.

Em um outro exemplo, nós podemos passar um outro contexto de outra temática para verificar a capacidade do modelo em identificar a resposta.

In [ ]:
context = """
Art. 2º  Poderão solicitar a realização de atividades acompanhadas estudante regularmente matriculado em curso, na modalidade presencial, de: educação profissional técnica de nível médio, de educação profissional tecnológica de graduação e de graduação, da Universidade Tecnológica Federal do Paraná - UTFPR, que atenda a seguinte condição:
I - estudante portador de afecções congênitas ou adquiridas, infecções, traumatismo ou outras condições mórbitas, determinando distúrbios agudos ou agudizados, caracterizados por incapacidade física relativa, incompatível com a frequência às aulas, desde que o estudante declare conservar suas condições intelectuais e emocionais necessárias para o prosseguimento das atividades escolares em novos moldes;
II - estudante em estado de gravidez, a partir do 8º (oitavo) mês de gestação, por no máximo 3 (três) meses; e estudante pelo nascimento/adoção por, no máximo, 7 (sete) dias;
III - estudante, como representante oficial do Brasil, dos estados-membros, dos municípios ou da UTFPR, em congressos científicos, em atividades de competição técnica/científica ou em competições artísticas ou desportivas de âmbito regional, estadual, nacional ou internacional.
"""
question = 'Quem pode solicitar atividades acompanhadas?'

result = nlp(question=question, context=context)

print(f"Answer: '{result['answer']}', score: {round(result['score'], 4)}, start: {result['start']}, end: {result['end']}")

Answer: 'estudante regularmente matriculado em curso', score: 0.5605, start: 68, end: 111


## Aplicação do modelo de perguntas e respostas

Para avaliar o uso desse modelo em um cenário mais complexo, vamos elaborar uma estratégia de perguntas e respostas considerando Regulamentos da UTFPR, novamente. Para isso, vamos fazer o download dos regulamentos para utilizarmos nesse exemplo.

Para simplificar, vamos focar em um modelo de perguntas e repostas que utilizará apenas um regulamento da UTFPR, de atividades acompanhadas, abonos de faltas e compensação de faltas.

In [ ]:
import io, tarfile, requests, os, pandas as pd


# download the dataset
def download (url, filename=''):
  if (os.path.isfile(filename)):
    print('Arquivo já existente no Runtime... Tudo OK')
    return
  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

urls = [
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/aa-ab-cf-dispensa-2021.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/ac-2022.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/diretrizes-grad-2022.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/ead-2022.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/estagio-2020.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/extensao-2022.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/rodp-2019.html",
#  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/tcc-2022.html"
]

filenames = []

for url in urls:
  filename = url.split('/')[-1]
  filenames.append(filename)
  download(url, filename)



Arquivo já existente no Runtime... Tudo OK


Após realizarmos o download do regulamento, vamos conduzir as mesmas etapas de extração dos textos, utilizando a biblioteca BeatifulSoup, e estratégias de pré-processamento dos dados.

In [ ]:
import codecs

from bs4 import BeautifulSoup


corpus = []

for filename in filenames:
  with codecs.open(filename, encoding='cp1252') as f:
    html = f.read()
    soup = BeautifulSoup(html)
    ps = soup.select('div[unselectable=on] ~ p')
    article = ''

    for p in ps:
      if p.get_text().lower().startswith('art.'):
        article = p.get_text()
        corpus.append(article)
      else:
        paragraph = p.get_text()
        corpus.append(f'{article} {paragraph}')


print('\n - '.join([ doc[:100] for doc in corpus[:10]]))

 ANEXO DA RESOLUÇÃO COGEP/UTFPR Nº 110, DE 19 DE OUTUBRO DE 2021.
 -   
 -  Regulamento para 
as atividades acompanhadas, o abono de faltas, a compensação de faltas, 
a compen
 -   
 -   
 -  Capítulo I
 -  Das Atividades Acompanhadas
 - Art. 1º  As atividades acompanhadas 
caracterizam-se pela execução em condições específicas, de ativ
 - Art. 2º  Poderão solicitar a 
realização de atividades acompanhadas estudante regularmente matricula
 - Art. 2º  Poderão solicitar a 
realização de atividades acompanhadas estudante regularmente matricula


In [ ]:
def clean(doc):
  words = doc.split()
  chars_to_replace = '!"#$%&\'()*+,-:;<=>?@[\\]^_`{|}~'
  table = doc.maketrans(chars_to_replace, ' ' * len(chars_to_replace))
  cleaned_words = [w.translate(table) for w in words]
  cleaned_doc = ' '.join(cleaned_words)
  cleaned_doc = cleaned_doc.replace(u'\xa0', u' ')
  cleaned_doc = cleaned_doc.replace(u'\u200b', u' ')
  cleaned_doc = cleaned_doc.replace(u'\n', u' ')
  cleaned_doc = cleaned_doc.lstrip()

  return cleaned_doc


corpus = [ clean(doc) for doc in corpus ]
corpus = [ doc for doc in corpus if len(doc.lstrip()) > 0 ]
print('\n - '.join([ doc[:50] for doc in corpus[:10]]))



ANEXO DA RESOLUÇÃO COGEP/UTFPR Nº 110  DE 19 DE OU
 - Regulamento para as atividades acompanhadas  o abo
 - Capítulo I
 - Das Atividades Acompanhadas
 - Art. 1º As atividades acompanhadas caracterizam se
 - Art. 2º Poderão solicitar a realização de atividad
 - Art. 2º Poderão solicitar a realização de atividad
 - Art. 2º Poderão solicitar a realização de atividad
 - Art. 2º Poderão solicitar a realização de atividad
 - Art. 3º A solicitação do regime de atividades acom


Após essas etapas, o nosso corpus é composto por diferentes documentos, cada um representando um artigo/parágrafo do regulamento da UTFPR.

Para tentar generalizar as respostas, vamos utilizar uma estratégia simplificada de extração de informações, utilizando TF-IDF para, dada uma pergunta, procurar pelos documentos que tem maior similaridade. Por fim, vamos utilizar o pipeline carregado para identificar as respostas às perguntas.

In [ ]:
import numpy as np


from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)


def get_answers (question):
  answers = []
  contexts = []

  [question_vec] = vectorizer.transform([question])
  for i, row in enumerate(X):
    dist = np.linalg.norm(question_vec.toarray() - row.toarray())
    contexts.append({ 'dist': dist, 'context': corpus[i] })

  contexts = sorted(contexts, key=lambda x: x['dist'])

  for context in contexts[:10]:
    result = nlp(question=question, context=context['context'])
    result['context'] = context['context']
    answers.append(result)

  answers = sorted(answers, key=lambda x: x['score'], reverse=True)

  return answers[:5]


questions = [
#    'Qual é o prazo de jubilamento do meu curso?',
#    'Quantas horas de atividades complementares eu preciso fazer?',
#    'Em qual período eu posso fazer estágio obrigatório?',
    'Como eu faço para solicitar compensação de faltas?',
    'Quem pode solicitar compensação de faltas?',
    'Quem define as atividades da compensação de faltas?',

    'Quando posso solicitar atividades acompanhadas?',
    'Em quais condições eu posso solicitar atividades acompanhadas?',
    'Qual é o prazo para que eu possa solicitar atividades acompanhadas?',
    'Eu posso solicitar atividades acompanhadas se eu estiver grávida?',
    'Posso solicitar atividades acompanhadas para participar de competições?',
    'Motivos religiosos permitem que eu solicite atividades acompanhadas?',
    'Em quais condições eu posso solicitar abono de faltas?',

#    'Como é calculado o meu coeficiente?',
#    'Qual é o prazo de cancelamento de matrícula?',
#    'Posso faltar mais que 25%?'
]

for question in questions:
  answers = get_answers(question)
  print('-----------\n')
  print(f'Question: {question}')
  for answer in answers:
    print(f'  - Answer: {answer["answer"]} ({answer["score"]}) - {answer["context"]}')

-----------

Question: Como eu faço para solicitar compensação de faltas?
  - Answer: atividades propostas pelo professor (0.5808660387992859) - Art. 19. Terá direito à compensação de faltas o estudante que se enquadra nas condições descritas no art. 2º  incisos I ou III desta Resolução  quando o período de afastamento for menor que o previsto no art. 4º desta Resolução. § 2º A compensação de faltas deve ocorrer por meio de atividades propostas pelo professor.
  - Answer: motivos religiosos (0.4581817090511322) - Regulamento para as atividades acompanhadas  o abono de faltas  a compensação de faltas  a compensação de faltas por motivos religiosos  as dispensas de frequência e o lançamento de faltas para estudante regularmente matriculado em curso  na modalidade presencial  de educação profissional técnica de nível médio  de educação profissional tecnológica de graduação e de graduação  da Universidade Tecnológica Federal do Paraná   UTFPR.
  - Answer: o coordenador do curso encaminhará